# Análise de SLA — Capacidade Operacional

Este notebook analisa o cumprimento de SLA (Service Level Agreement) da equipe de suporte técnico.

Perguntas respondidas:
- Qual a taxa global de cumprimento de SLA?
- Existe diferença de cumprimento entre prioridades (High / Medium / Low)?
- Quais analistas mais e menos cumprem o SLA?
- Quais tópicos apresentam maior risco de violação?
- A taxa de cumprimento melhorou ou piorou ao longo do tempo?

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.data_processing import tratar_dados

df = tratar_dados()
df.head()

## 1. Preparação dos Dados para SLA

Filtramos apenas tickets **Closed** (com data de fechamento definida) e criamos a coluna `cumpriu_sla`, que indica se o ticket foi resolvido antes do prazo contratual.

In [ ]:
# Apenas tickets fechados com prazo SLA definido
df_fechados = df[
    (df['status_ticket'] == 'Closed') &
    (df['data_fechamento'].notna()) &
    (df['sla_previsto_resolucao'].notna())
].copy()

# Coluna booleana de cumprimento
df_fechados['cumpriu_sla'] = df_fechados['data_fechamento'] <= df_fechados['sla_previsto_resolucao']

print(f"Total de tickets fechados com SLA: {len(df_fechados)}")
print(f"Cumpriram SLA: {df_fechados['cumpriu_sla'].sum()}")
print(f"Violaram SLA: {(~df_fechados['cumpriu_sla']).sum()}")
df_fechados[['prioridade', 'analista_responsavel', 'data_fechamento', 'sla_previsto_resolucao', 'cumpriu_sla']].head(10)

## 2. Taxa Global de Cumprimento de SLA

In [ ]:
taxa_global = df_fechados['cumpriu_sla'].mean() * 100
print(f"Taxa global de cumprimento de SLA: {taxa_global:.1f}%")
print(f"Taxa de violação: {100 - taxa_global:.1f}%")

## 3. SLA por Prioridade

Tickets de alta prioridade tendem a ter prazos mais curtos. Avaliamos se o cumprimento reflete essa pressão.

In [ ]:
sla_prioridade = (
    df_fechados.groupby('prioridade')['cumpriu_sla']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'taxa_cumprimento', 'count': 'total_tickets'})
    .sort_values('taxa_cumprimento', ascending=False)
)
sla_prioridade['taxa_cumprimento_pct'] = (sla_prioridade['taxa_cumprimento'] * 100).round(1)

print(sla_prioridade[['total_tickets', 'taxa_cumprimento_pct']])

fig, ax = plt.subplots(figsize=(8, 5))
cores = {'High': '#d62728', 'Medium': '#ff7f0e', 'Low': '#2ca02c'}
ordem = ['High', 'Medium', 'Low']
valores = [sla_prioridade.loc[p, 'taxa_cumprimento_pct'] if p in sla_prioridade.index else 0 for p in ordem]
bars = ax.bar(ordem, valores, color=[cores[p] for p in ordem], edgecolor='white', linewidth=1.5)
ax.axhline(taxa_global, color='gray', linestyle='--', linewidth=1.2, label=f'Média geral: {taxa_global:.1f}%')
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Taxa de Cumprimento de SLA por Prioridade', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Prioridade')
ax.set_ylabel('% Tickets dentro do SLA')
ax.legend()
plt.tight_layout()
plt.show()

## 4. SLA por Analista

Identificamos quais analistas têm maior dificuldade em cumprir o SLA — informação complementar ao tempo médio de resolução.

In [ ]:
sla_analista = (
    df_fechados.groupby('analista_responsavel')['cumpriu_sla']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'taxa_cumprimento', 'count': 'total_tickets'})
    .sort_values('taxa_cumprimento', ascending=False)
)
sla_analista['taxa_cumprimento_pct'] = (sla_analista['taxa_cumprimento'] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sla_analista.index, sla_analista['taxa_cumprimento_pct'],
               color='#4472C4', edgecolor='white', linewidth=1)
ax.axvline(taxa_global, color='#d62728', linestyle='--', linewidth=1.5, label=f'Média geral: {taxa_global:.1f}%')
for bar, val in zip(bars, sla_analista['taxa_cumprimento_pct']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2, f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlim(0, 115)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Taxa de Cumprimento de SLA por Analista', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('% Tickets dentro do SLA')
ax.legend()
plt.tight_layout()
plt.show()

print(sla_analista[['total_tickets', 'taxa_cumprimento_pct']])

## 5. SLA por Tópico

Categorias de tickets que mais violam o SLA indicam onde a operação tem maior pressão de resolução.

In [ ]:
sla_topico = (
    df_fechados.groupby('topico')['cumpriu_sla']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'taxa_cumprimento', 'count': 'total_tickets'})
    .sort_values('taxa_cumprimento', ascending=True)
)
sla_topico['taxa_cumprimento_pct'] = (sla_topico['taxa_cumprimento'] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
cores_topico = ['#d62728' if v < taxa_global else '#2ca02c' for v in sla_topico['taxa_cumprimento_pct']]
bars = ax.barh(sla_topico.index, sla_topico['taxa_cumprimento_pct'],
               color=cores_topico, edgecolor='white', linewidth=1)
ax.axvline(taxa_global, color='gray', linestyle='--', linewidth=1.5, label=f'Média geral: {taxa_global:.1f}%')
for bar, val in zip(bars, sla_topico['taxa_cumprimento_pct']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2, f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlim(0, 115)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Taxa de Cumprimento de SLA por Tópico', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('% Tickets dentro do SLA')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Heatmap: Analista × Prioridade

Visão cruzada que revela em qual combinação analista/prioridade ocorrem as maiores concentrações de violação.

In [ ]:
pivot_heatmap = (
    df_fechados.groupby(['analista_responsavel', 'prioridade'])['cumpriu_sla']
    .mean()
    .unstack(fill_value=0) * 100
)
# Ordenar colunas por prioridade lógica
ordem_col = [c for c in ['High', 'Medium', 'Low'] if c in pivot_heatmap.columns]
pivot_heatmap = pivot_heatmap[ordem_col]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot_heatmap,
    annot=True, fmt='.1f', cmap='RdYlGn',
    vmin=0, vmax=100,
    linewidths=0.5, ax=ax,
    cbar_kws={'label': '% Cumprimento SLA'}
)
ax.set_title('Cumprimento de SLA (%) — Analista × Prioridade', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Prioridade')
ax.set_ylabel('Analista')
plt.tight_layout()
plt.show()

## 7. Evolução Temporal do Cumprimento de SLA

Verificamos se a taxa de cumprimento melhorou, piorou ou se manteve estável ao longo do tempo.

In [ ]:
df_fechados['mes_ano'] = df_fechados['data_fechamento'].dt.to_period('M')

sla_temporal = (
    df_fechados.groupby('mes_ano')['cumpriu_sla']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'taxa_cumprimento', 'count': 'total_tickets'})
)
sla_temporal['taxa_cumprimento_pct'] = sla_temporal['taxa_cumprimento'] * 100

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

meses = sla_temporal.index.astype(str)
ax1.plot(meses, sla_temporal['taxa_cumprimento_pct'], color='#4472C4', linewidth=2.5, marker='o', markersize=5, label='% SLA cumprido')
ax1.axhline(taxa_global, color='#4472C4', linestyle='--', linewidth=1, alpha=0.5, label=f'Média global: {taxa_global:.1f}%')
ax1.fill_between(meses, taxa_global, sla_temporal['taxa_cumprimento_pct'],
                  where=sla_temporal['taxa_cumprimento_pct'] >= taxa_global,
                  alpha=0.1, color='#2ca02c', label='Acima da média')
ax1.fill_between(meses, taxa_global, sla_temporal['taxa_cumprimento_pct'],
                  where=sla_temporal['taxa_cumprimento_pct'] < taxa_global,
                  alpha=0.1, color='#d62728', label='Abaixo da média')

ax2.bar(meses, sla_temporal['total_tickets'], alpha=0.2, color='gray', label='Volume tickets')

ax1.set_ylim(0, 110)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_title('Evolução Mensal da Taxa de Cumprimento de SLA', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Mês')
ax1.set_ylabel('% SLA cumprido', color='#4472C4')
ax2.set_ylabel('Volume de tickets', color='gray')
plt.xticks(rotation=45, ha='right')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Conclusão — SLA

Resumo dos achados desta análise:

In [ ]:
melhor_analista = sla_analista['taxa_cumprimento_pct'].idxmax()
pior_analista = sla_analista['taxa_cumprimento_pct'].idxmin()
melhor_prioridade = sla_prioridade['taxa_cumprimento_pct'].idxmax()
pior_prioridade = sla_prioridade['taxa_cumprimento_pct'].idxmin()
pior_topico = sla_topico['taxa_cumprimento_pct'].idxmin()

print("=" * 55)
print("       RESUMO — ANÁLISE DE SLA")
print("=" * 55)
print(f"  Taxa global de cumprimento : {taxa_global:.1f}%")
print(f"  Melhor prioridade          : {melhor_prioridade} ({sla_prioridade.loc[melhor_prioridade, 'taxa_cumprimento_pct']:.1f}%)")
print(f"  Pior prioridade            : {pior_prioridade} ({sla_prioridade.loc[pior_prioridade, 'taxa_cumprimento_pct']:.1f}%)")
print(f"  Melhor analista            : {melhor_analista} ({sla_analista.loc[melhor_analista, 'taxa_cumprimento_pct']:.1f}%)")
print(f"  Pior analista              : {pior_analista} ({sla_analista.loc[pior_analista, 'taxa_cumprimento_pct']:.1f}%)")
print(f"  Tópico mais crítico        : {pior_topico} ({sla_topico.loc[pior_topico, 'taxa_cumprimento_pct']:.1f}%)")
print("=" * 55)